## Semantic Search Quick Start

In [1]:
# setup the embedding model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

/home/tushar/Desktop/blank/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8734.07it/s]


### Initialize the Elasticsearch client

In [2]:
from elasticsearch import Elasticsearch
from getpass import getpass

# Create the client instance
client = Elasticsearch(
    # For local development
    hosts=["http://localhost:9200"]
)

### Enable Telemetry
Knowing that you are using this notebook helps us decide where to invest our efforts to improve our products. We would like to ask you that you run the following code to let us gather anonymous usage statistics. See telemetry.py for details. Thank you!


In [3]:
!curl -O -s https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/telemetry/telemetry.py
from telemetry import enable_telemetry

client = enable_telemetry(client, "00-quick-start")

Telemetry enabled for "00-quick-start". Thank you!


In [8]:
print(client.info())

{'name': '2ae99bf23950', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'QmXcrAAhRH2KElx_ozJbrg', 'version': {'number': '8.6.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'f67ef2df40237445caa70e2fef79471cc608d70d', 'build_date': '2023-01-04T09:35:21.782467981Z', 'build_snapshot': False, 'lucene_version': '9.4.2', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [9]:
# INdex some test data
# Our client is set up and connected to our Elastic deployment. Now we need some data to test out the basics of Elasticsearch queries. We'll use a small index of books with the following fields:

# title
# authors
# publish_date
# num_reviews
# publisher

In [13]:
# create an index
client.indices.delete(index='book_index', ignore_unavailable=True)
print("NOTE: at any time you can come back to this section and run the delete function above to remove your index and start from scratch.")

NOTE: at any time you can come back to this section and run the delete function above to remove your index and start from scratch.


In [14]:
# define the mapping
mappings = {
    "properties":{
        "title_vector":{
            "type":"dense_vector",
            "dims":384,
            "index":"true",
            "similarity":"cosine"
        }
    }
}

# Create the index
client.indices.create(index='book_index', mappings=mappings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'book_index'})

In [15]:
# index the test data
import json
from urllib.request import urlopen

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/notebooks/search/data.json"
response = urlopen(url)
books = json.loads(response.read())

operations = []
for book in books:
    operations.append({"index": {"_index": "book_index"}})
    # transforming the title into an embedding using the model
    book['title_vector'] = model.encode(book['title']).tolist()
    operations.append(book)
client.bulk(index="book_index", operations=operations, refresh=True)

ObjectApiResponse({'took': 116, 'errors': False, 'items': [{'index': {'_index': 'book_index', '_id': '2swncZ8BhBsZ8gCCwdHq', '_version': 1, 'result': 'created', 'forced_refresh': True, '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1, 'status': 201}}, {'index': {'_index': 'book_index', '_id': '28wncZ8BhBsZ8gCCwdHq', '_version': 1, 'result': 'created', 'forced_refresh': True, '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 1, '_primary_term': 1, 'status': 201}}, {'index': {'_index': 'book_index', '_id': '3MwncZ8BhBsZ8gCCwdHq', '_version': 1, 'result': 'created', 'forced_refresh': True, '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 2, '_primary_term': 1, 'status': 201}}, {'index': {'_index': 'book_index', '_id': '3cwncZ8BhBsZ8gCCwdHq', '_version': 1, 'result': 'created', 'forced_refresh': True, '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 3, '_primary_term': 1, 'status': 201}}, {'index': {'_

In [16]:
# Aside: pretty printing Elasticsearch responses

def pretty_response(response):
    if len(response['hits']['hits']) == 0:
        print("Your search returned no results.")
    else:
        for hit in response['hits']['hits']:
            id = hit["_id"]
            publication_date = hit["_source"]["publish_date"]
            score = hit["_score"]
            title = hit["_source"]["title"]
            summary = hit["_source"]["summary"]
            publisher = hit["_source"]["publisher"]
            num_reviews = hit["_source"]["num_reviews"]
            authors = hit["_source"]["authors"]
            pretty_output = f"\nID: {id}\nPublication date: {publication_date}\nTitle: {title}\nSummary: {summary}\nPublisher: {publisher}\nReviews: {num_reviews}\nAuthors: {authors}\nScore: {score}"
            print(pretty_output) 

In [17]:
# making queries

response = client.search(
    index="book_index",
    knn={
        "field":"title_vector",
        "query_vector":model.encode("Python"),
        "k":10,
        "num_candidates":100,
    }
)

In [19]:
pretty_response(response)


ID: 28wncZ8BhBsZ8gCCwdHq
Publication date: 2019-05-03
Title: Python Crash Course
Summary: A fast-paced, no-nonsense guide to programming in Python
Publisher: no starch press
Reviews: 42
Authors: ['eric matthes']
Score: 0.84482634

ID: 2swncZ8BhBsZ8gCCwdHq
Publication date: 2019-10-29
Title: The Pragmatic Programmer: Your Journey to Mastery
Summary: A guide to pragmatic programming for software engineers and developers
Publisher: addison-wesley
Reviews: 30
Authors: ['andrew hunt', 'david thomas']
Score: 0.64513975

ID: 48wncZ8BhBsZ8gCCwdHq
Publication date: 2012-06-27
Title: Introduction to the Theory of Computation
Summary: Introduction to the theory of computation and complexity theory
Publisher: cengage learning
Reviews: 33
Authors: ['michael sipser']
Score: 0.6339196

ID: 3MwncZ8BhBsZ8gCCwdHq
Publication date: 2020-04-06
Title: Artificial Intelligence: A Modern Approach
Summary: Comprehensive introduction to the theory and practice of artificial intelligence
Publisher: pearson
Revi